# Notebook 03: Upload to HuggingFace & Deploy Demo

Upload DPO LoRA adapter to HuggingFace Hub and deploy Gradio demo.

**Requirements**: HuggingFace token, DPO model from Notebook 02

## Setup

In [ ]:
!pip install -q transformers peft huggingface_hub gradio torch bitsandbytes accelerate

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/smallllm-dpo'
DRIVE_DIR = '/content/drive/MyDrive/smallllm-dpo'

if os.path.exists(DRIVE_DIR):
    !cp -r {DRIVE_DIR} {PROJECT_DIR}
os.chdir(PROJECT_DIR)

import sys
sys.path.insert(0, PROJECT_DIR)

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
except:
    import getpass
    hf_token = getpass.getpass('HuggingFace token: ')

login(token=hf_token)

## Part 1: Upload DPO Adapter to HuggingFace

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
DPO_MODEL_PATH = './dpo_output/sft_dpo'
REPO_ID = 'Cheng-1/qwen2.5-3b-fc-dpo'  # Update with your HF username

# Create repo if needed
api.create_repo(REPO_ID, exist_ok=True, repo_type='model')

# Upload adapter files
api.upload_folder(
    folder_path=DPO_MODEL_PATH,
    repo_id=REPO_ID,
    commit_message='Upload DPO LoRA adapter for function calling alignment',
)
print(f'Adapter uploaded to: https://huggingface.co/{REPO_ID}')

In [ ]:
# Upload model card
MODEL_CARD = """
---
language: en
license: apache-2.0
tags:
  - dpo
  - function-calling
  - qlora
  - qwen2.5
  - alignment
base_model: Qwen/Qwen2.5-3B-Instruct
---

# Qwen2.5-3B Function Calling DPO Adapter

QLoRA DPO adapter for function calling alignment on Qwen2.5-3B-Instruct.

## Training
- Method: DPO (Direct Preference Optimization) with QLoRA 4-bit
- Data: 893 synthetic preference pairs (chosen/rejected function calls)
- Beta: 0.5, LoRA rank: 16, 1 epoch, lr: 5e-6

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
model = PeftModel.from_pretrained(base, "Cheng-1/qwen2.5-3b-fc-dpo")
```

## Part of SmallLLM Research Series
- Phase 1: [small-llms-tool-use](https://github.com/XIECHENG6/small-llms-tool-use) — SFT for Function Calling
- Phase 6: [smallllm-dpo](https://github.com/XIECHENG6/smallllm-dpo) — DPO Alignment
"""

with open('/tmp/README.md', 'w') as f:
    f.write(MODEL_CARD)

api.upload_file(
    path_or_fileobj='/tmp/README.md',
    path_in_repo='README.md',
    repo_id=REPO_ID,
)
print('Model card uploaded')

## Part 2: Upload Preference Dataset

In [ ]:
DATASET_REPO = 'Cheng-1/fc-preference-data'  # Update with your HF username

api.create_repo(DATASET_REPO, exist_ok=True, repo_type='dataset')

api.upload_file(
    path_or_fileobj='data/preference_pairs.jsonl',
    path_in_repo='preference_pairs.jsonl',
    repo_id=DATASET_REPO,
    repo_type='dataset',
    commit_message='Upload function calling preference pairs for DPO training',
)
print(f'Dataset uploaded to: https://huggingface.co/datasets/{DATASET_REPO}')

## Part 3: Gradio Demo

In [ ]:
import torch
import json
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from src.data.function_pool import FUNCTION_POOL
from src.data.prompt_templates import build_fc_system_prompt, build_chat_messages
from src.evaluation.metrics import parse_function_call

# Load DPO model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct', trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-3B-Instruct', quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True,
)
dpo_model = PeftModel.from_pretrained(base_model, './dpo_output/sft_dpo')
dpo_model.eval()
print('DPO model loaded')

In [ ]:
from src.data.function_pool import get_functions_by_names

def predict(query, function_names_str):
    func_names = [n.strip() for n in function_names_str.split(',') if n.strip()]
    functions = get_functions_by_names(func_names)
    if not functions:
        functions = FUNCTION_POOL[:5]

    system_prompt = build_fc_system_prompt(functions)
    messages = build_chat_messages(system_prompt, query)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(dpo_model.device)

    with torch.no_grad():
        outputs = dpo_model.generate(
            **inputs, max_new_tokens=256,
            do_sample=False, pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    parsed = parse_function_call(text)
    if parsed:
        return json.dumps(parsed, indent=2, ensure_ascii=False)
    return f'Raw output (failed to parse):\n{text}'

demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Textbox(label='User Query', placeholder='e.g., What is the weather in Tokyo?'),
        gr.Textbox(label='Available Functions (comma-separated)', value='get_weather, search_web, get_news'),
    ],
    outputs=gr.Code(label='Function Call (JSON)', language='json'),
    title='SmallLLM-DPO: Function Calling with Preference Alignment',
    description='Qwen2.5-3B fine-tuned with DPO for accurate function calling.',
    examples=[
        ['What is the weather like in Shanghai today?', 'get_weather, search_web, get_news'],
        ['Search flights from Beijing to Tokyo on July 1st', 'search_flights, book_hotel, get_directions'],
        ['Translate hello world to Chinese', 'translate_text, summarize_text, search_web'],
        ['How much is 100 USD in CNY?', 'convert_currency, get_exchange_rate, get_stock_price'],
    ],
)

demo.launch(share=True)

In [ ]:
# Upload results to Drive
!cp -r results {DRIVE_DIR}/
print('All outputs saved to Drive')